# Proyecto 2 — Análisis Inicial y Selección de Problema

**Etapa:** Parte I — Búsqueda, EDA inicial y Selección de Problema  

---

##Datasets analizados

| # | Dataset | Fuente | Dominio | Problema |
|---|---------|--------|---------|----------|
| 1 | Bank Customer Churn | Kaggle | Banca / Finanzas | Clasificación |
| 2 | Heart Disease | Kaggle | Salud / Medicina | Clasificación |
| 3 | Video Game Sales + Metacritic | Kaggle | Entretenimiento | Regresión |
| 4 | Most Streamed Spotify Songs 2024 | Kaggle | Música / Streaming | Regresión |

---


# Diccionario de Columnas — Proyecto 2

---

##  Dataset 1 — Bank Customer Churn
**Archivo:** `Customer-Churn-Records.csv` | **Filas:** 10.000 | **Columnas:** 18

| # | Columna (original) | Traducción | Tipo | Descripción |
|---|---|---|---|---|
| 1 | RowNumber | Número de fila | int | Número de registro. No tiene efecto en la predicción |
| 2 | CustomerId | ID de cliente | int | Identificador único del cliente. No influye en el churn |
| 3 | Surname | Apellido | str | Apellido del cliente. No tiene impacto en la decisión de abandonar |
| 4 | CreditScore | Puntaje crediticio | int | Puntaje de crédito del cliente. Mayor puntaje = menor probabilidad de abandono |
| 5 | Geography | País | str | País de residencia del cliente (France, Spain, Germany) |
| 6 | Gender | Género | str | Género del cliente (Male = Masculino / Female = Femenino) |
| 7 | Age | Edad | int | Edad del cliente. Clientes mayores tienden a ser más leales |
| 8 | Tenure | Antigüedad | int | Años que el cliente lleva en el banco. Mayor antigüedad = más lealtad |
| 9 | Balance | Saldo en cuenta | float | Saldo disponible en la cuenta. Saldos altos se asocian a menor churn |
| 10 | NumOfProducts | N° de productos | int | Cantidad de productos bancarios contratados por el cliente |
| 11 | HasCrCard | ¿Tiene tarjeta de crédito? | int | Indica si posee tarjeta de crédito (1 = Sí / 0 = No) |
| 12 | IsActiveMember | ¿Es miembro activo? | int | Los clientes activos son menos propensos a abandonar (1 = Sí / 0 = No) |
| 13 | EstimatedSalary | Salario estimado | float | Salario estimado del cliente. Salarios bajos se asocian a mayor churn |
| 14 | **Exited** | **¿Abandonó el banco?** | int | **Variable objetivo: 1 = abandonó el banco / 0 = permanece** |
| 15 | Complain | ¿Presentó reclamo? | int | Indica si el cliente realizó alguna queja (1 = Sí / 0 = No) |
| 16 | Satisfaction Score | Puntuación de satisfacción | int | Calificación del cliente sobre la resolución de su reclamo (1–5) |
| 17 | Card Type | Tipo de tarjeta | str | Tipo de tarjeta de crédito que posee el cliente (DIAMOND, GOLD, SILVER, PLATINUM) |
| 18 | Point Earned | Puntos acumulados | int | Puntos de fidelidad ganados mediante el uso de la tarjeta de crédito |

---

---
# Sección 0 — Configuración del entorno

Lo primero que debemos hacer es conectarnos al drive para poder acceder a los dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Librerías
import os, glob, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

In [ ]:
#  Función reutilizable: resumen de nulos
def resumen_nulos(df, titulo=''):
    null_df = pd.DataFrame({
        'Nulos': df.isnull().sum(),
        '% Nulos': (df.isnull().sum() / len(df) * 100).round(1)
    }).sort_values('% Nulos', ascending=False)
    null_df = null_df[null_df['Nulos'] > 0]
    if null_df.empty:
        print(f'{titulo}: Sin valores nulos.')
        return null_df
    print(f'\n Nulos en {titulo}:')
    print(null_df.to_string())
    return null_df

# Función reutilizable: outliers IQR
def resumen_outliers(df, cols, nombre=''):
    rows = []
    for col in cols:
        if col not in df.columns: continue
        data = df[col].dropna()
        Q1, Q3 = data.quantile(0.25), data.quantile(0.75)
        IQR = Q3 - Q1
        n = ((data < Q1 - 1.5*IQR) | (data > Q3 + 1.5*IQR)).sum()
        rows.append({
            'Variable': col, 'Q1': round(Q1, 2), 'Q3': round(Q3, 2),
            'IQR': round(IQR, 2), 'N Outliers': n,
            '% Outliers': round(n / len(data) * 100, 2)
        })
    result = pd.DataFrame(rows)
    if not result.empty:
        print(f'\n [{nombre}] Resumen de outliers (método IQR):')
        display(result)
    return result


---
# Dataset 1 — Bank Customer Churn
**Fuente:** Kaggle — `radheshyamkollipara/bank-customer-churn`  
**Dominio:** Banca / Finanzas  
**Problema:** Clasificación binaria:  predecir si un cliente abandonará el banco

## 1.1 Descripción del conjunto de datos

Dataset de **10.000 clientes** de un banco multinacional con variables demográficas, financieras y de comportamiento.  
El target `Exited` indica si el cliente abandonó el banco (1) o permanece (0).

Existen algunas columnas que en verdad no nos interesan:


|  Columna | Descripción |
|---|---|
| RowNumber | Número de registro. No tiene efecto en la predicción |
|  CustomerId | Identificador único del cliente. No influye en el churn |
|  Surname |  Apellido del cliente. No tiene impacto en la decisión de abandonar |

por lo que las eliminaremos de inmediato al cargar el dataset

## 1.2 Descarga y carga de datos

In [ ]:
df_bank=pd.read_csv('/content/drive/MyDrive/Colab Notebooks/datasets/Customer-Churn-Records.csv')
df_bank.drop(columns=[c for c in ['RowNumber','CustomerId','Surname'] if c in df_bank.columns], inplace=True)
print(f' Dataset cargado: {df_bank.shape[0]:,} filas × {df_bank.shape[1]} columnas')
print(f'Columnas: {df_bank.columns.tolist()}')
df_bank.head()

## 1.3 Análisis estadístico descriptivo

In [ ]:
print(f'Informacion del dataset: \n')
df_bank.info()

print(f'\nPrimeras 5 filas: \n\n {df_bank.head()}\n')
print(f'Últimas 5 filas: \n\n {df_bank.tail()}\n')
print(f'Cantidad de elementos duplicados:\n\n {df_bank.duplicated()}\n')
print(f'Cantidad de elementos nulos:\n \n{df_bank.isnull().sum()}\n')
print(f'Tipos de datos :\n \n {df_bank.dtypes.to_string()}')

In [ ]:
print('\n Estadísticas numéricas ')
#esta forma de mostrar las estadisticas de un modo mas "agradable", viene de la IA
# lo encontre más ordenado y quice aplicarlo.
#se puede apreciar que las variables( columnas) estan ordenadas en vertical
#y no horizontal como lo hace normalmente el .describe()
num1 = df_bank.select_dtypes(include=np.number)
desc1 = num1.describe().T
desc1['skewness'] = num1.skew().round(3)
desc1['cv'] = (desc1['std'] / desc1['mean']).round(3)
display(desc1.style.background_gradient(cmap='Blues', subset=['mean','std']))


print('\n Variables categóricas')
for col in df_bank.select_dtypes(include='object').columns:
    print(df_bank[col].value_counts(normalize=True).map(lambda x: f'{x:.1%}').to_string())
    print('\n')

In [ ]:
nulos1 = resumen_nulos(df_bank, 'Bank Customer Churn')

if not nulos1.empty:
    plt.figure(figsize=(8,4))
    nulos1['% Nulos'].sort_values().plot(kind='barh', color='#D85A30')
    plt.title('% de nulos por columna'); plt.tight_layout(); plt.show()

print('\n Resumen de outliers (IQR):')
all_num1 = df_bank.select_dtypes(include=np.number).columns.tolist()
print(resumen_outliers(df_bank, all_num1).to_string(index=False))

In [ ]:
#Veremos si existe alguna inconsistencia con los valores de algunas columnas
print(df_bank['Geography'].unique())
print(df_bank['Gender'].unique())
print(df_bank['Card Type'].unique())

## 1.4 Visualizaciones
### 1.4.1 Distribución del target y desbalance de clases

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts1 = df_bank['Exited'].value_counts()
axes[0].bar(['No Churn (0)', 'Churn (1)'], counts1.values,
            color=['#5DCAA5','#D85A30'], edgecolor='white')
for i, v in enumerate(counts1.values):
    axes[0].text(i, v+80, f'{v:,}', ha='center', fontweight='bold')
axes[0].set_title('Distribución absoluta del target')
axes[0].set_ylabel('N° de clientes')

axes[1].pie(counts1.values, labels=['No Churn','Churn'],
            autopct='%1.1f%%', colors=['#5DCAA5','#D85A30'],
            startangle=90, wedgeprops={'edgecolor':'white'})
axes[1].set_title('Proporción de clases')

plt.suptitle('DS1 — Desbalance de clases (target: Exited)', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()
print(f'Tasa de churn: {df_bank["Exited"].mean():.1%}')

### 1.4.2 Histogramas de variables numéricas por churn

In [ ]:
num_cols1 = [c for c in df_bank.select_dtypes(include=np.number).columns if c != 'Exited']
nrows = (len(num_cols1)+2)//3
fig, axes = plt.subplots(nrows, 3, figsize=(15, nrows*3.2))
axes = axes.flatten()

for i, col in enumerate(num_cols1):
    ax = axes[i]
    for val, color, lbl in [(0,'#5DCAA5','No Churn'),(1,'#D85A30','Churn')]:
        ax.hist(df_bank[df_bank['Exited']==val][col], bins=30, alpha=0.6,
                color=color, label=lbl, density=True)
    ax.set_title(col, fontsize=10)
    ax.legend(fontsize=7)
    ax.set_xlabel(f'skew: {df_bank[col].skew():.2f}', fontsize=8)
for j in range(i+1, len(axes)): axes[j].set_visible(False)

plt.suptitle('DS1 — Distribución de variables numéricas por Churn', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

### 1.4.3 Tasa de churn por variables categóricas

In [ ]:
plot_cats1 = ['Geography','Gender','Card Type','IsActiveMember','NumOfProducts','Satisfaction Score']
plot_cats1 = [c for c in plot_cats1 if c in df_bank.columns]
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(plot_cats1):
    ax = axes[i]
    rate = df_bank.groupby(col)['Exited'].mean().sort_values(ascending=False) * 100
    bars = ax.bar(rate.index.astype(str), rate.values,
                  color=sns.color_palette('muted', len(rate)), edgecolor='white')
    for bar in bars:
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                f'{bar.get_height():.1f}%', ha='center', fontsize=9)
    ax.set_title(f'Churn rate por {col}', fontsize=10)
    ax.set_ylabel('Churn rate (%)')
    ax.set_ylim(0, rate.max()*1.3)

plt.suptitle('DS1 — Churn rate por variable categórica', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

### 1.4.4 Box plots — Detección de outliers

In [ ]:
box_cols1 = ['CreditScore','Age','Balance','EstimatedSalary','Point Earned']
box_cols1 = [c for c in box_cols1 if c in df_bank.columns]
fig, axes = plt.subplots(1, len(box_cols1), figsize=(15, 5))

for i, col in enumerate(box_cols1):
    ax = axes[i]
    ax.boxplot([df_bank[df_bank['Exited']==0][col], df_bank[df_bank['Exited']==1][col]],
               labels=['No Churn','Churn'], patch_artist=True,
               boxprops=dict(facecolor='#E6F1FB'),
               medianprops=dict(color='#D85A30', linewidth=2))
    ax.set_title(col, fontsize=9)

plt.suptitle('DS1 — Box Plots por Churn', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

print(resumen_outliers(df_bank, box_cols1).to_string(index=False))

### 1.4.5 Mapa de calor de correlaciones

In [ ]:
df_bank_enc = df_bank.copy()
for col in df_bank_enc.select_dtypes(include='object').columns:
    df_bank_enc[col] = pd.Categorical(df_bank_enc[col]).codes
corr1 = df_bank_enc.corr()

plt.figure(figsize=(13, 9))
mask = np.triu(np.ones_like(corr1, dtype=bool))
sns.heatmap(corr1, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, linewidths=0.5, linecolor='white', annot_kws={'size':8})
plt.title('DS1 — Mapa de correlaciones (Bank Customer Churn)', fontsize=13, fontweight='bold')
plt.xticks(rotation=45, ha='right'); plt.tight_layout(); plt.show()

print('\n Correlaciones con Exited (target):')
print(corr1['Exited'].drop('Exited').sort_values(key=abs, ascending=False).head(10).map(lambda x: f'{x:.3f}'))

## 1.5 Resumen de hallazgos — Dataset 1

| Aspecto | Detalle |
|---|---|
| **Tamaño** | 10.000 filas × 15 columnas útiles |
| **Nulos** |  Sin valores nulos |
| **Desbalance** |  casi 20% Churn vs un 80% No Churn, el dataset requiere SMOTE o `class_weight` |
| **Outliers** | Moderados en `CreditScore` y `Age`; balance bimodal (muchos en $0) |
| **Correlación clave** | `Complain` altamente correlacionado con `Exited`, hay que tener precaucion ante un posible data leakage |
| **Skewness** | `Balance` bimodal; `EstimatedSalary` casi uniforme |
| **Problemática** | **Clasificación binaria:** predecir abandono de cliente |
| **Relevancia** | Retener un cliente cuesta 5–7x menos que adquirir uno nuevo |

#Proyecto 2 Parte II

**Preprocesamiento y Optimización**

Objetivo: Realizar el preprocesamiento de datos y la optimización de modelos de machine learning para el conjunto de datos seleccionado (`Customer-Churn-Records`). La meta es elegir la técnica de machine learning más adecuada y optimizar sus hiperparámetros para obtener el mejor rendimiento posible.

In [ ]:
!pip install lightgbm xgboost optuna imbalanced-learn

In [ ]:
#volvemos a cargar los imports ya que agregué algunos
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Preprocesamiento y modelado
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_auc_score, roc_curve, f1_score,
                              precision_score, recall_score, accuracy_score,
                              ConfusionMatrixDisplay)

# Modelos
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# Balanceo de clases
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

# Optimización de hiperparámetros
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
import optuna
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)


---
# Sección 1 — Carga y Limpieza de Datos

En la Parte I ya identificamos que el dataset **no presenta valores nulos** y que las columnas
`RowNumber`, `CustomerId` y `Surname` no aportan valor predictivo, por lo que se eliminan
desde la carga.  
También detectamos outliers moderados en `CreditScore` y `Age`, y un desbalance de clases
aproximado de 80/20 (No Churn / Churn).

In [ ]:
df_bank.head()

##1.1 Informacion Actual del Dataset

In [ ]:
#asi esta actualmente el dataset despues de la parte 1
print(f'Informacion del dataset: \n')
df_bank.info()

print(f'\nPrimeras 5 filas: \n\n {df_bank.head()}\n')
print(f'Últimas 5 filas: \n\n {df_bank.tail()}\n')
print(f'Cantidad de elementos duplicados:\n\n {df_bank.duplicated()}\n')
print(f'Cantidad de elementos nulos:\n \n{df_bank.isnull().sum()}\n')
print(f'Tipos de datos :\n \n {df_bank.dtypes.to_string()}')

print('\n Estadísticas numéricas ')
num1 = df_bank.select_dtypes(include=np.number)
desc1 = num1.describe().T
desc1['skewness'] = num1.skew().round(3)
desc1['cv'] = (desc1['std'] / desc1['mean']).round(3)
display(desc1.style.background_gradient(cmap='Blues', subset=['mean','std']))


print('\n Variables categóricas')
for col in df_bank.select_dtypes(include='object').columns:
    print(df_bank[col].value_counts(normalize=True).map(lambda x: f'{x:.1%}').to_string())
    print('\n')

NO tenemos ni nulos ni duplicados

## 1.2 Tratamiento de Outliers

Del EDA anterior sabemos que `CreditScore` y `Age` tienen outliers por IQR.
En un dataset bancario, valores extremos de edad o score crediticio son perfectamente
posibles (clientes muy jóvenes con tarjeta, adultos mayores, scores extremos por historial
crediticio específico), por lo que **no es posible eliminarlos** sino que los contenemos con
**capping al percentil 1–99**, preservando la señal sin distorsionar la distribución.

In [ ]:
#funcion para hacerle capping a los outliers
def cap_outliers(df, cols, lower=0.01, upper=0.99):
    df_bank_out = df_bank.copy()
    for col in cols:
        lo = df_bank_out[col].quantile(lower)
        hi = df_bank_out[col].quantile(upper)
        antes = ((df_bank_out[col] < lo) | (df_bank_out[col] > hi)).sum()
        df_bank_out[col] = df_bank_out[col].clip(lo, hi)
        print(f"  {col:20s} → {antes} valores ajustados  |  rango: [{lo:.1f}, {hi:.1f}]")
    return df_bank_out

outlier_cols = ['CreditScore', 'Age']
print("Capping de outliers (P1–P99):")
df_bank = cap_outliers(df_bank, outlier_cols)

---
# Sección 2 — Preparación de Features y División del Dataset

## 2.1 Separación de features y target

Separamos X (features) de y (target `Exited`) y dividimos en entrenamiento (80%) y prueba (20%)
con estratificación para mantener la proporción de clases en ambos conjuntos.

In [ ]:
target = 'Exited'
X = df_bank.drop(columns=[target])
y = df_bank[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape[0]:,} filas | Test: {X_test.shape[0]:,} filas")
print(f"Proporción churn — Train: {y_train.mean():.1%} | Test: {y_test.mean():.1%}")

## 2.2 Identificación de columnas por tipo

Clasificamos las columnas según el tipo de transformación que requieren:
- **Numéricas continuas**: escala con `StandardScaler`
- **Categóricas nominales**: codificación con `OneHotEncoder`
- **Binarias / ordinales ya numéricas**: pasan sin transformación o solo se escalan segun sea el caso.


In [ ]:
# Columnas categóricas (object)
cat_cols = X.select_dtypes(include='object').columns.tolist()

# Columnas numéricas (todas las demás)
num_cols = X.select_dtypes(include=np.number).columns.tolist()

print(f"Columnas categóricas ({len(cat_cols)}): {cat_cols}")
print(f"Columnas numéricas   ({len(num_cols)}): {num_cols}")

---
# Sección 3 — Construcción del Pipeline de Preprocesamiento

Usamos `ColumnTransformer` para aplicar transformaciones diferenciadas por tipo de columna,
encapsulado dentro de un `Pipeline` de sklearn. Esto garantiza:
- Reproducibilidad: el mismo proceso se aplica en train y test
- Prevención de data leakage: el scaler se ajusta solo con datos de entrenamiento
- Facilidad para integrar con `GridSearchCV` / `RandomizedSearchCV` / `Optuna`



## 3.1 Definición del preprocesador

In [ ]:
# Sub-pipeline para numéricas: imputación (por seguridad) y escalado
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Sub-pipeline para categóricas: imputación y  OHE
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# ColumnTransformer que une ambos sub-pipelines
preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, num_cols),
    ('cat', categorical_transformer, cat_cols)
])

print("Preprocesador definido")
print(f"{len(num_cols)} columnas numéricas: StandardScaler")
print(f"{len(cat_cols)} columnas categóricas: OneHotEncoder")

## 3.2 Inspección del preprocesamiento

Verificamos que el pipeline transforma correctamente los datos antes de continuar con el modelado.

In [ ]:
# Ajuste solo con train, transformación a ambos conjuntos
X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc  = preprocessor.transform(X_test)

# Nombres de columnas resultantes
ohe_cols = preprocessor.named_transformers_['cat']['onehot'].get_feature_names_out(cat_cols)
all_feature_names = num_cols + list(ohe_cols)

print(f"Shape train procesado: {X_train_proc.shape}")
print(f"Shape test  procesado: {X_test_proc.shape}")
print(f"Total features después de OHE: {len(all_feature_names)}")

# Vista rápida del resultado
pd.DataFrame(X_train_proc, columns=all_feature_names).head(3)

---
# Sección 4 — Entrenamiento Inicial y Comparación de Modelos

## 4.1 Estrategia ante el desbalance de clases

El dataset tiene casi un de 20% churn vs un 80% de no-churn. Para manejar este desbalance usamos dos estrategias:
1. **`class_weight='balanced'`** en modelos que lo soporten (LR, DT, RF)
2. **SMOTE** como oversample en el pipeline de entrenamiento (para comparar)

En esta sección comparamos primero sin SMOTE (solo `class_weight`) para tener una línea base limpia.


## 4.2 Definición de modelos candidatos

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

modelos = {
    'Logistic Regression': Pipeline([
        ('pre', preprocessor),
        ('clf', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))
    ]),
    'KNN': Pipeline([
        ('pre', preprocessor),
        ('clf', KNeighborsClassifier(n_neighbors=7))
    ]),
    'Decision Tree': Pipeline([
        ('pre', preprocessor),
        ('clf', DecisionTreeClassifier(class_weight='balanced', max_depth=8, random_state=42))
    ]),
    'Random Forest': Pipeline([
        ('pre', preprocessor),
        ('clf', RandomForestClassifier(class_weight='balanced', n_estimators=200,
                                        random_state=42, n_jobs=-1))
    ]),
    'XGBoost': Pipeline([
        ('pre', preprocessor),
        ('clf', XGBClassifier(scale_pos_weight=4, eval_metric='logloss',
                               random_state=42, n_jobs=-1))
    ]),
    'LightGBM': Pipeline([
        ('pre', preprocessor),
        ('clf', LGBMClassifier(class_weight='balanced', random_state=42,
                                n_jobs=-1, verbose=-1))
    ]),
}

print(f"{len(modelos)} modelos definidos: {list(modelos.keys())}")

##4.3 Evaluación con validación cruzada

In [ ]:
resultados = []

for nombre, pipe in modelos.items():
    roc_scores  = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='roc_auc',   n_jobs=-1)
    f1_scores   = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='f1',        n_jobs=-1)
    acc_scores  = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='accuracy',  n_jobs=-1)

    resultados.append({
        'Modelo': nombre,
        'ROC-AUC (mean)': roc_scores.mean().round(4),
        'ROC-AUC (std)':  roc_scores.std().round(4),
        'F1 (mean)':      f1_scores.mean().round(4),
        'F1 (std)':       f1_scores.std().round(4),
        'Accuracy (mean)':acc_scores.mean().round(4),
    })

df_resultados = pd.DataFrame(resultados).sort_values('ROC-AUC (mean)', ascending=False)
display(df_resultados.style.background_gradient(cmap='Greens',
        subset=['ROC-AUC (mean)', 'F1 (mean)', 'Accuracy (mean)']))

## 4.4 Visualización comparativa de modelos

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
metrics = ['ROC-AUC (mean)', 'F1 (mean)', 'Accuracy (mean)']
colors = sns.color_palette('muted', len(df_resultados))

for ax, metric in zip(axes, metrics):
    df_plot = df_resultados.sort_values(metric)
    bars = ax.barh(df_plot['Modelo'], df_plot[metric], color=colors, edgecolor='white')
    for bar in bars:
        ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
                f'{bar.get_width():.4f}', va='center', fontsize=9)
    ax.set_title(metric, fontweight='bold')
    ax.set_xlim(0, df_plot[metric].max() * 1.12)

plt.suptitle('Comparación de modelos — Validación cruzada (5-fold)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

Al revisar los resultados de la validación cruzada, todos los modelos obtuvieron métricas superiores a 0.99 en ROC-AUC, F1 y Accuracy. Este rendimiento es **irreal** para un problema de churn bancario y es una señal clara de **data leakage**.

La variable Complain es la causa. Como se identificó en la Parte I, esta columna indica si el cliente realizó una queja, pero en la práctica ese registro ocurre al mismo tiempo o después de que el cliente ya decidió abandonar el banco. Es decir, el modelo no está aprendiendo a predecir el churn, sino que está detectando una consecuencia de él. En producción, esta información no estaría disponible en el momento en que se necesita hacer la predicción.

Por esta razón, voy a eliminar a Complain del conjunto de features antes de cualquier entrenamiento. Esta decisión sacrifica métricas artificialmente altas a cambio de un modelo que refleja un escenario real y que puede generar valor genuino para el negocio.

In [ ]:
X = df_bank.drop(columns=[target, 'Complain'])

* Desde este punto volvere a ejecutar lo que se habia hecho pero ahora excluyendo la columna `Complain`

In [ ]:
# Columnas categóricas (object)
cat_cols = X.select_dtypes(include='object').columns.tolist()

# Columnas numéricas (todas las demás)
num_cols = X.select_dtypes(include=np.number).columns.tolist()

print(f"Columnas categóricas ({len(cat_cols)}): {cat_cols}")
print(f"Columnas numéricas   ({len(num_cols)}): {num_cols}")

In [ ]:
# Sub-pipeline para numéricas: imputación (por seguridad) y escalado
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Sub-pipeline para categóricas: imputación y  OHE
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# ColumnTransformer que une ambos sub-pipelines
preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, num_cols),
    ('cat', categorical_transformer, cat_cols)
])

print("Preprocesador definido")
print(f"{len(num_cols)} columnas numéricas: StandardScaler")
print(f"{len(cat_cols)} columnas categóricas: OneHotEncoder")

In [ ]:
# Ajuste solo con train, transformación a ambos conjuntos
X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc  = preprocessor.transform(X_test)

# Nombres de columnas resultantes
ohe_cols = preprocessor.named_transformers_['cat']['onehot'].get_feature_names_out(cat_cols)
all_feature_names = num_cols + list(ohe_cols)

print(f"Shape train procesado: {X_train_proc.shape}")
print(f"Shape test  procesado: {X_test_proc.shape}")
print(f"Total features después de OHE: {len(all_feature_names)}")

# Vista rápida del resultado
pd.DataFrame(X_train_proc, columns=all_feature_names).head(3)

In [ ]:
#Definicion de modelos
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

modelos = {
    'Logistic Regression': Pipeline([
        ('pre', preprocessor),
        ('clf', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))
    ]),
    'KNN': Pipeline([
        ('pre', preprocessor),
        ('clf', KNeighborsClassifier(n_neighbors=7))
    ]),
    'Decision Tree': Pipeline([
        ('pre', preprocessor),
        ('clf', DecisionTreeClassifier(class_weight='balanced', max_depth=8, random_state=42))
    ]),
    'Random Forest': Pipeline([
        ('pre', preprocessor),
        ('clf', RandomForestClassifier(class_weight='balanced', n_estimators=200,
                                        random_state=42, n_jobs=-1))
    ]),
    'XGBoost': Pipeline([
        ('pre', preprocessor),
        ('clf', XGBClassifier(scale_pos_weight=4, eval_metric='logloss',
                               random_state=42, n_jobs=-1))
    ]),
    'LightGBM': Pipeline([
        ('pre', preprocessor),
        ('clf', LGBMClassifier(class_weight='balanced', random_state=42,
                                n_jobs=-1, verbose=-1))
    ]),
}

print(f"{len(modelos)} modelos definidos: {list(modelos.keys())}")

In [ ]:
#validacion cruzada
resultados = []

for nombre, pipe in modelos.items():
    roc_scores  = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='roc_auc',   n_jobs=-1)
    f1_scores   = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='f1',        n_jobs=-1)
    acc_scores  = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='accuracy',  n_jobs=-1)

    resultados.append({
        'Modelo': nombre,
        'ROC-AUC (mean)': roc_scores.mean().round(4),
        'ROC-AUC (std)':  roc_scores.std().round(4),
        'F1 (mean)':      f1_scores.mean().round(4),
        'F1 (std)':       f1_scores.std().round(4),
        'Accuracy (mean)':acc_scores.mean().round(4),
    })

df_resultados = pd.DataFrame(resultados).sort_values('ROC-AUC (mean)', ascending=False)
display(df_resultados.style.background_gradient(cmap='Greens',
        subset=['ROC-AUC (mean)', 'F1 (mean)', 'Accuracy (mean)']))

In [ ]:
#visualizacion comparativa de los modelos (despues de elminar complain)
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
metrics = ['ROC-AUC (mean)', 'F1 (mean)', 'Accuracy (mean)']
colors = sns.color_palette('muted', len(df_resultados))

for ax, metric in zip(axes, metrics):
    df_plot = df_resultados.sort_values(metric)
    bars = ax.barh(df_plot['Modelo'], df_plot[metric], color=colors, edgecolor='white')
    for bar in bars:
        ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
                f'{bar.get_width():.4f}', va='center', fontsize=9)
    ax.set_title(metric, fontweight='bold')
    ax.set_xlim(0, df_plot[metric].max() * 1.12)

plt.suptitle('Comparación de modelos — Validación cruzada (5-fold)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

ahora si podemos hablar de un resultado mas real

## 4.5 Selección del modelo para optimización

Basándonos en las métricas de validación cruzada, seleccionamos el modelo con mejor
**ROC-AUC** y **F1-Score**, que son las métricas más relevantes en un problema de churn
con desbalance de clases.  

**Random Forest** y **LightGBM** son los candidatos más fuertes en este dataset.

Priorizamos **LightGBM** por su eficiencia y buen manejo de variables categóricas y desbalanceo.

  - Mejor ROC-AUC en validación cruzada
  - Robusto ante desbalance de clases con class_weight='balanced'
  - Entrenamiento rápido lo que permite GridSearch exhaustivo
  - Nativo para datos mixtos (numéricos y OHE)

---
# Sección 5 — Optimización de Hiperparámetros

Implementamos tres estrategias de búsqueda de hiperparámetros de menor a mayor sofisticación:

`GridSearchCV` , `RandomizedSearchCV` , `Optuna`.


## 5.1 GridSearchCV — Búsqueda exhaustiva

In [ ]:
from scipy.stats import randint, uniform

# Pipeline base para GridSearch
pipe_gs = Pipeline([
    ('pre', preprocessor),
    ('clf', LGBMClassifier(class_weight='balanced', random_state=42,
                            n_jobs=-1, verbose=-1))
])

param_grid = {
    'clf__n_estimators':   [100, 200, 300],
    'clf__max_depth':      [4, 6, 8],
    'clf__learning_rate':  [0.05, 0.1],
    'clf__num_leaves':     [31, 63],
}

gs = GridSearchCV(
    pipe_gs, param_grid, cv=cv,
    scoring='roc_auc', n_jobs=-1, verbose=0
)
gs.fit(X_train, y_train)

print("GridSearchCV completado")
print(f"  Mejores parámetros : {gs.best_params_}")
print(f"  Mejor ROC-AUC (CV) : {gs.best_score_:.4f}")

## 5.2 RandomizedSearchCV — Búsqueda aleatoria en espacio amplio

In [ ]:
pipe_rs = Pipeline([
    ('pre', preprocessor),
    ('clf', LGBMClassifier(class_weight='balanced', random_state=42,
                            n_jobs=-1, verbose=-1))
])

param_dist = {
    'clf__n_estimators':    randint(100, 500),
    'clf__max_depth':       randint(3, 12),
    'clf__learning_rate':   uniform(0.01, 0.2),
    'clf__num_leaves':      randint(20, 150),
    'clf__min_child_samples': randint(10, 50),
    'clf__subsample':       uniform(0.6, 0.4),
    'clf__colsample_bytree':uniform(0.6, 0.4),
}

rs = RandomizedSearchCV(
    pipe_rs, param_dist,
    n_iter=50, cv=cv,
    scoring='roc_auc', n_jobs=-1,
    random_state=42, verbose=0
)
rs.fit(X_train, y_train)

print("RandomizedSearchCV completado")
print(f"  Mejores parámetros : {rs.best_params_}")
print(f"  Mejor ROC-AUC (CV) : {rs.best_score_:.4f}")

## 5.3 Optuna — Optimización bayesiana (TPE Sampler)

In [ ]:
def objective(trial):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 100, 600),
        'max_depth':         trial.suggest_int('max_depth', 3, 12),
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'num_leaves':        trial.suggest_int('num_leaves', 20, 200),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 60),
        'subsample':         trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha':         trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda':        trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
    }
    pipe = Pipeline([
        ('pre', preprocessor),
        ('clf', LGBMClassifier(**params, class_weight='balanced',
                                random_state=42, n_jobs=-1, verbose=-1))
    ])
    score = cross_val_score(pipe, X_train, y_train, cv=cv,
                             scoring='roc_auc', n_jobs=-1).mean()
    return score

study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
study.optimize(objective, n_trials=80, show_progress_bar=True)

print("\nOptuna completado")
print(f"  Mejor ROC-AUC (CV): {study.best_value:.4f}")
print(f"  Mejores parámetros: {study.best_params}")

## 5.4 Visualización de la evolución de Optuna

In [ ]:
# Evolución del score en cada trial
trial_scores = [t.value for t in study.trials]
best_so_far  = pd.Series(trial_scores).cummax()

plt.figure(figsize=(12, 4))
plt.plot(trial_scores, alpha=0.4, color='#5DCAA5', label='Score por trial')
plt.plot(best_so_far,  color='#D85A30', linewidth=2.5, label='Mejor acumulado')
plt.xlabel('Trial')
plt.ylabel('ROC-AUC (CV)')
plt.title('Optuna — Evolución de la optimización (LightGBM)', fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()

---
# Sección 6 — Evaluación del Modelo Optimizado

Comparamos el rendimiento del modelo base vs. el optimizado con Optuna en el conjunto
de **test** (datos que ningún modelo ha visto durante el proceso de optimización).

## 6.1 Entrenamiento del modelo final con mejores hiperparámetros

In [ ]:
# Modelo con Optuna best params
best_params_optuna = study.best_params

pipe_final = Pipeline([
    ('pre', preprocessor),
    ('clf', LGBMClassifier(**best_params_optuna, class_weight='balanced',
                            random_state=42, n_jobs=-1, verbose=-1))
])
pipe_final.fit(X_train, y_train)

# Modelo base (sin optimizar)
pipe_base = Pipeline([
    ('pre', preprocessor),
    ('clf', LGBMClassifier(class_weight='balanced', random_state=42,
                            n_jobs=-1, verbose=-1))
])
pipe_base.fit(X_train, y_train)

print("Modelos entrenados con datos de entrenamiento completos")

## 6.2 Comparación de métricas — Base vs Optimizado

In [ ]:
def evaluar_modelo(pipe, X_test, y_test, nombre):
    y_pred  = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]
    return {
        'Modelo':     nombre,
        'Accuracy':   round(accuracy_score(y_test, y_pred),  4),
        'Precision':  round(precision_score(y_test, y_pred), 4),
        'Recall':     round(recall_score(y_test, y_pred),    4),
        'F1-Score':   round(f1_score(y_test, y_pred),        4),
        'ROC-AUC':    round(roc_auc_score(y_test, y_proba),  4),
    }

metricas_base    = evaluar_modelo(pipe_base,  X_test, y_test, 'LightGBM Base')
metricas_optuna  = evaluar_modelo(pipe_final, X_test, y_test, 'LightGBM Optuna')

# También incluimos GridSearch y RandomSearch
pipe_gs_final = gs.best_estimator_
pipe_rs_final = rs.best_estimator_
metricas_gs = evaluar_modelo(pipe_gs_final, X_test, y_test, 'LightGBM GridSearch')
metricas_rs = evaluar_modelo(pipe_rs_final, X_test, y_test, 'LightGBM RandomSearch')

df_comp = pd.DataFrame([metricas_base, metricas_gs, metricas_rs, metricas_optuna])
display(df_comp.style.highlight_max(subset=['Accuracy','Precision','Recall','F1-Score','ROC-AUC'],
                                     color='blue'))

## 6.3 Curvas ROC comparativas

In [ ]:
plt.figure(figsize=(8, 6))
for pipe, label in [(pipe_base, 'Base'), (pipe_gs_final, 'GridSearch'),
                     (pipe_rs_final, 'RandomSearch'), (pipe_final, 'Optuna')]:
    y_proba = pipe.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    plt.plot(fpr, tpr, label=f'{label} (AUC = {auc:.4f})', linewidth=2)

plt.plot([0,1],[0,1],'--', color='gray', label='Random')
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('Curvas ROC — LightGBM Base vs Optimizado', fontweight='bold')
plt.legend(loc='lower right'); plt.tight_layout(); plt.show()

Podemos ver que la mejora en la optimizacion del modelo base es casi un 0.01 en AUC, es bajo pero consistente lo que indica que la optimización aportó valor real sin ser espectacular.

Tambien podemos ver que los tres metodos llegan a un punto similar, esto sugiere que el modelo ya esta cerca de su techo con estas features disponibles.

Podemos ver que **RandomSearch** le gano por muy poco **Optuna** (una diferencia de 0.003), entonces podemos afirmar que para este dataset la exploracion aleatoria puede estar en competencia, lo que también sugiere que con más trials, Optuna probablemente superaría a RandomSearch, ya que su búsqueda se vuelve más eficiente a medida que acumula información de los trials anteriores.

## 6.4 Matriz de confusión — Modelo final (Optuna)

In [ ]:
y_pred_final = pipe_final.predict(X_test)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_final, display_labels=['No Churn', 'Churn'],
    cmap='Blues', ax=axes[0]
)
axes[0].set_title('Matriz de Confusión — LightGBM Optuna')

# Normalizada
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_final, display_labels=['No Churn', 'Churn'],
    cmap='Blues', normalize='true', ax=axes[1]
)
axes[1].set_title('Matriz de Confusión Normalizada')

plt.tight_layout(); plt.show()
print("\nReporte de clasificación:")
print(classification_report(y_test, y_pred_final, target_names=['No Churn', 'Churn']))

En churn bancario el error más costoso es el Falso Negativo (perder un cliente sin haberlo detectado), porque retener un cliente cuesta 5–7 veces menos que adquirir uno nuevo. Con esa lógica, un 76% de recall es aceptable pero hay margen de mejora, y una estrategia válida sería bajar el umbral de clasificación de 0.5 a algo como 0.4 para capturar más churns a costa de más falsos positivos.

## 6.5 Importancia de features — LightGBM Optuna

In [ ]:
lgbm_model = pipe_final.named_steps['clf']
ohe_cols_   = pipe_final.named_steps['pre'].named_transformers_['cat']['onehot'].get_feature_names_out(cat_cols)
feature_names = num_cols + list(ohe_cols_)

importances = pd.Series(lgbm_model.feature_importances_, index=feature_names)
top_feat = importances.sort_values(ascending=False).head(15)

plt.figure(figsize=(10, 6))
top_feat.sort_values().plot(kind='barh', color=sns.color_palette('muted', 15))
plt.title('Top 15 features — LightGBM Optuna', fontweight='bold')
plt.xlabel('Importancia')
plt.tight_layout(); plt.show()

---
# Sección 7 — Resumen y Conclusiones

## 7.1 Comparación de estrategias de optimización

| Método | Exploración | Tiempo estimado | Notas |
|--------|-------------|-----------------|-------|
| **GridSearchCV** | Exhaustiva (espacio fijo) | Medio | Ideal para espacios pequeños y bien acotados |
| **RandomizedSearchCV** | Aleatoria (espacio amplio) | Rápido | Buen equilibrio eficiencia/cobertura |
| **Optuna (TPE)** | Bayesiana más Pruning | Variable (80 trials) | Mayor precisión en espacios continuos grandes |

## 7.2 Hallazgos principales

| Aspecto | Detalle |
|---------|---------|
| **Preprocesamiento** | Pipeline con `ColumnTransformer`: `StandardScaler` en numéricas, `OneHotEncoder` en categóricas |
| **Desbalance** | Manejado con `class_weight='balanced'` en todos los modelos |
| **Mejor modelo base** | LightGBM superó a LR, KNN, DT, RF y XGBoost en ROC-AUC |
| **Mejora con Optuna** | Incremento en ROC-AUC y F1-Score respecto al modelo base |
| **Feature más importante** | `Complain` (alta correlación con churn vista en Parte I) |
| **Riesgo data leakage** | `Complain` fue identificado como variable de riesgo; su alta importancia confirma la sospecha |
|Mejora de Análisis| Al eliminar `Complain` podemos analizar un modelo más realista y contemplar verdaderamente la variacion en los modelos|